# Colab2: T4 GPU Training and Ensemble Prediction

This notebook is for Google Colab with a `T4 GPU`.

It will:
1. Verify GPU runtime
2. Clone the project
3. Install dependencies
4. Upload the new dataset
5. Optionally upload the ISOT-trained checkpoint zip
6. Train a new model on `Fake or Real News Dataset`
7. Evaluate it
8. Run ensemble prediction with two checkpoints


## Step 0: Select T4 GPU

In Colab:
- `Runtime` -> `Change runtime type`
- Hardware accelerator -> `T4 GPU`
- Save


In [ ]:
import os
import sys
import torch

print('Torch version:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('GPU is not enabled. Switch Colab runtime to T4 GPU first.')

print('GPU:', torch.cuda.get_device_name(0))
print('GPU memory (GB):', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2))

PROJECT_DIR = '/content/FND_2027'
print('Project dir:', PROJECT_DIR)


In [ ]:
!git clone https://github.com/au510621104021/FND_2027.git /content/FND_2027
%cd /content/FND_2027
!git branch --show-current
!git log -1 --oneline


In [ ]:
!pip install -q --upgrade pip
!pip install -q -r requirements.txt
print('Dependencies installed')


## Step 1: Prepare folders

This notebook trains the second model on `Fake or Real News Dataset`.

Expected files:
- `train.csv`
- `test.csv`

Optional ISOT model zip:
- `trained_model.zip`


In [ ]:
import os

SECOND_DATASET_DIR = os.path.join(PROJECT_DIR, 'Fake or Real News Dataset')
CHECKPOINT_DIR = os.path.join(PROJECT_DIR, 'checkpoints')
SECOND_CHECKPOINT_DIR = os.path.join(CHECKPOINT_DIR, 'fake_or_real')

os.makedirs(SECOND_DATASET_DIR, exist_ok=True)
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(SECOND_CHECKPOINT_DIR, exist_ok=True)

print('Second dataset dir:', SECOND_DATASET_DIR)
print('Checkpoint dir:', CHECKPOINT_DIR)
print('Second checkpoint dir:', SECOND_CHECKPOINT_DIR)


## Step 2: Upload `train.csv` and `test.csv`

Run the next cell, upload the files for `Fake or Real News Dataset`, and they will be copied into the correct project folder.


In [ ]:
from google.colab import files
import shutil

uploaded = files.upload()

for name in uploaded.keys():
    dest = os.path.join(SECOND_DATASET_DIR, name)
    shutil.move(name, dest)
    print('Saved:', dest)

print('Final files:', sorted(os.listdir(SECOND_DATASET_DIR)))


In [ ]:
required = ['train.csv', 'test.csv']
missing = [name for name in required if not os.path.exists(os.path.join(SECOND_DATASET_DIR, name))]
if missing:
    raise FileNotFoundError(f'Missing dataset files: {missing}')
print('Dataset files are ready')


## Step 3: Optional - Upload your ISOT-trained model zip

If you want to use ensemble prediction in Colab, upload `trained_model.zip` here.

If you skip this step, training of the new model still works.


In [ ]:
UPLOAD_ISOT_ZIP = False
print('Set UPLOAD_ISOT_ZIP = True if you want to upload the old ISOT checkpoint zip now.')


In [ ]:
import zipfile
import shutil

if UPLOAD_ISOT_ZIP:
    uploaded = files.upload()
    zip_names = [name for name in uploaded.keys() if name.lower().endswith('.zip')]
    if not zip_names:
        raise FileNotFoundError('No zip file uploaded.')

    isot_zip = zip_names[0]
    print('Uploaded zip:', isot_zip)

    with zipfile.ZipFile(isot_zip, 'r') as zf:
        zf.extractall(PROJECT_DIR)

    print('Extracted zip into project directory')
    print('Checkpoint files:', os.listdir(CHECKPOINT_DIR))
else:
    print('Skipping ISOT checkpoint upload')


## Step 4: Tune config for T4 GPU

This cell updates the second-dataset config for Colab.


In [ ]:
import yaml

config_path = os.path.join(PROJECT_DIR, 'config', 'config_fake_or_real.yaml')
with open(config_path, 'r', encoding='utf-8') as f:
    config = yaml.safe_load(f)

config['data']['data_dir'] = SECOND_DATASET_DIR
config['data'].pop('data_dirs', None)
config['data']['num_workers'] = 2
config['data']['pin_memory'] = True

config['training']['batch_size'] = 4
config['training']['fp16'] = True
config['training']['num_epochs'] = 10

config['logging']['checkpoint_dir'] = './checkpoints/fake_or_real'
config['logging']['log_dir'] = './logs/fake_or_real'
config['inference']['model_checkpoint'] = './checkpoints/fake_or_real/best_model.pt'
config['publication']['results_dir'] = './results/fake_or_real'

with open(config_path, 'w', encoding='utf-8') as f:
    yaml.safe_dump(config, f, sort_keys=False)

print('Updated config at:', config_path)
print('Batch size:', config['training']['batch_size'])
print('Epochs:', config['training']['num_epochs'])
print('Data dir:', config['data']['data_dir'])


## Step 5: Train the new model on T4 GPU

This will create:
- `checkpoints/fake_or_real/best_model.pt`
- `checkpoints/fake_or_real/latest_model.pt`


In [ ]:
%cd /content/FND_2027
!python scripts/train.py --config config/config_fake_or_real.yaml --device cuda


## Step 6: Evaluate the new model


In [ ]:
%cd /content/FND_2027
!python scripts/evaluate.py --checkpoint checkpoints/fake_or_real/best_model.pt --config config/config_fake_or_real.yaml --device cuda --ablation --bootstrap --latex


## Step 7: Upload an image for prediction

You can test:
- the old ISOT model alone
- the new model alone
- both models together with ensemble averaging


In [ ]:
uploaded = files.upload()
image_name = next(iter(uploaded.keys()))
IMAGE_PATH = os.path.join(PROJECT_DIR, image_name)
print('Image path:', IMAGE_PATH)


In [ ]:
TEXT_INPUT = 'Enter the news text that belongs to the uploaded image here.'
print(TEXT_INPUT)


## Step 8A: Predict with ISOT model only

Run this only if `checkpoints/best_model.pt` exists.


In [ ]:
!python scripts/predict.py --checkpoint checkpoints/best_model.pt --text "$TEXT_INPUT" --image "$IMAGE_PATH" --device cuda


## Step 8B: Predict with the new dataset model only


In [ ]:
!python scripts/predict.py --checkpoint checkpoints/fake_or_real/best_model.pt --text "$TEXT_INPUT" --image "$IMAGE_PATH" --device cuda


## Step 8C: Ensemble prediction with both models

Run this only if both checkpoints exist.


In [ ]:
!python scripts/predict_ensemble.py --text "$TEXT_INPUT" --image "$IMAGE_PATH" --checkpoints checkpoints/best_model.pt checkpoints/fake_or_real/best_model.pt --device cuda


## Step 9: Save trained second-model checkpoint

Download the new checkpoint after training.


In [ ]:
from google.colab import files

files.download('/content/FND_2027/checkpoints/fake_or_real/best_model.pt')


## Step 10: Export the trained second model as a zip

This creates a zip similar to your earlier trained-model archive.


In [ ]:
!rm -rf /content/FND_2027/export_fake_or_real
!mkdir -p /content/FND_2027/export_fake_or_real/checkpoints
!cp /content/FND_2027/checkpoints/fake_or_real/best_model.pt /content/FND_2027/export_fake_or_real/checkpoints/
!cp /content/FND_2027/checkpoints/fake_or_real/latest_model.pt /content/FND_2027/export_fake_or_real/checkpoints/
!cp /content/FND_2027/checkpoints/fake_or_real/training_history.json /content/FND_2027/export_fake_or_real/checkpoints/
%cd /content/FND_2027
!rm -f fake_or_real_trained_model.zip
!zip -r fake_or_real_trained_model.zip export_fake_or_real


In [ ]:
from google.colab import files
files.download('/content/FND_2027/fake_or_real_trained_model.zip')
